In [28]:
# augmentation version 3: here, ALL classed were augmented.
# happy was undersampled to 2330 (below our desired amount).
# all three classes generated new images and are now equal at 3500 images each.

In [29]:
from pathlib import Path
from collections import Counter

from capstone_data.image_loader import load_dataset

data_path = Path.home() / "Downloads" / "cleaned_data"

print("Dataset path:", data_path)
print("Dataset folder exists:", data_path.exists())

Dataset path: /Users/melissatobias/Downloads/cleaned_data
Dataset folder exists: True


In [30]:
from capstone_data.image_loader import load_dataset 
from collections import Counter

data_path = Path.home() / "Downloads" / "cleaned_data"

dataset = load_dataset (data_path)

print (f"Total images: {len (dataset)}")

print("\nImages per class:")
class_counts = Counter (item["class_name"] for item in dataset)

for class_name, count in class_counts. items():
    print (f"{class_name}: {count}")

if class_counts:
    print(f"\nTotal classes found: {len(class_counts)}")
    print(f"Largest class: {max(class_counts, key=class_counts.get)}")
    print(f"Smallest class: {min(class_counts, key=class_counts.get)}")

Total images: 16779

Images per class:
angry: 3089
happy: 6677
sad: 1207
neutral: 5806

Total classes found: 4
Largest class: happy
Smallest class: sad


In [31]:
# splitting 70-15-15

import pandas as pd
from sklearn.model_selection import train_test_split
from collections import Counter

# convert the loader's records into a DataFrame
df = pd.DataFrame(dataset)

# use each record's class name as its stratification label
X = df
y = df["class_name"].str.lower()

# reserve 15% for the final test set
train_val_df, test_df = train_test_split(
    df,
    test_size=0.15,
    random_state=42,
    stratify=y)

# reserve 15% of the original data for validation.
train_df, val_df = train_test_split(
    train_val_df,
    test_size=0.15 / 0.85,
    random_state=42,
    stratify=train_val_df["class_name"].str.lower())

print("Training images:", len(train_df))
print("Validation images:", len(val_df))
print("Test images:", len(test_df))
print("Total:", len(train_df) + len(val_df) + len(test_df))

Training images: 11745
Validation images: 2517
Test images: 2517
Total: 16779


In [32]:
# checking distributions after split

classes = ["angry", "happy", "sad"]

for split_name, split_df in [
    ("Training", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    print(f"\n{split_name} distribution:")

    counts = Counter(split_df["class_name"].str.lower())

    for class_name in classes:
        print(f"  {class_name.capitalize():<8}: {counts.get(class_name, 0)}")


Training distribution:
  Angry   : 2163
  Happy   : 4673
  Sad     : 845

Validation distribution:
  Angry   : 463
  Happy   : 1002
  Sad     : 181

Test distribution:
  Angry   : 463
  Happy   : 1002
  Sad     : 181


In [33]:

from pathlib import Path
from collections import Counter
from PIL import Image, ImageEnhance, ImageOps
import random

random.seed(42)

# only the three emotions (no neutral)
classes = ["angry", "happy", "sad"]
target = 3500

output = Path.home() / "Downloads" / "Capstone-DATA4381" / "augmentation" / "aug_data_v3"

if output.exists():
    raise FileExistsError(
        f"{output} already exists.")

output.mkdir(parents=True)

# filter training split to the three target emotions
train_3class = train_df[
    train_df["class_name"].str.lower().isin(classes)].copy()

print("Original training counts:")
for emotion in classes:
    count = (train_3class["class_name"].str.lower() == emotion).sum()
    print(f"{emotion.capitalize():<8}: {count}")

print("\nExport folder:", output)


Original training counts:
Angry   : 2163
Happy   : 4673
Sad     : 845

Export folder: /Users/melissatobias/Downloads/Capstone-DATA4381/augmentation/aug_data_v3


In [34]:

def augment_grayscale(image):
    image = image.convert("L")
    original_size = image.size

    # horizontal flip (mirrored)
    if random.random() < 0.5:
        image = ImageOps.mirror(image)

    # small rotation (random left or right)
    if random.random() < 0.7:
        angle = random.uniform(-13, 13)

        image = image.rotate(
            angle,
            resample=Image.Resampling.BICUBIC,
            expand=True,
            fillcolor=0)

    # brightness change
    if random.random() < 0.5:
        image = ImageEnhance.Brightness(image).enhance(
            random.uniform(0.9, 1.2))

    return image.convert("L")


In [35]:

for emotion in classes:
    class_df = train_3class[
        train_3class["class_name"].str.lower() == emotion].copy()
    
    undersampling_message = None
    # randomly undersample happy
    if emotion == "happy" and len(class_df) > target:
        original_happy_count = len(class_df)
        class_df = class_df.sample(
            n=2330,
            random_state=42)
        undersampling_message = (f"(Original Happy amount: {original_happy_count} - Reduced to: {len(class_df)} before augmentation)")

    class_df = class_df.reset_index(drop=True)

    class_folder = output/emotion
    class_folder.mkdir(parents=True, exist_ok=True)

    original_images = []

    for index, row in class_df.iterrows():
        with Image.open(row["path"]) as img:
            gray_img = img.convert("L").copy()

        original_images.append(gray_img)

        gray_img.save(
            class_folder / f"original_{index:05d}.jpg",
            format="JPEG",
            quality=95)

    # extra images only if the class is below 3500
    additional_needed = target - len(original_images)

    for index in range(additional_needed):
        source = random.choice(original_images)
        augmented = augment_grayscale(source)

        augmented.save(
            class_folder / f"augmented_{index:05d}.jpg",
            format="JPEG",
            quality=95)

    print(
        f"{emotion.capitalize()}: "
        f"{len(original_images)} originals + "
        f"{additional_needed} augmented = "
        f"{len(original_images) + additional_needed} images")

    if undersampling_message:
        print(undersampling_message)

print("\nExport saved to:", output)


Angry: 2163 originals + 1337 augmented = 3500 images
Happy: 2330 originals + 1170 augmented = 3500 images
(Original Happy amount: 4673 - Reduced to: 2330 before augmentation)
Sad: 845 originals + 2655 augmented = 3500 images

Export saved to: /Users/melissatobias/Downloads/Capstone-DATA4381/augmentation/aug_data_v3


In [36]:
print("Final dataset verification:")

for emotion in classes:
    folder = output/emotion

    files = [
        p for p in folder.iterdir()
        if p.suffix.lower() in {".jpg", ".jpeg", ".png"}]

    print(f"{emotion.capitalize():<8}: {len(files)} images")

    assert len(files) == target, (
        f"{emotion} does not have {target} images")

Final dataset verification:
Angry   : 3500 images
Happy   : 3500 images
Sad     : 3500 images
